# C-test ONE · Results-ECI

*Ctest-Results-ECI-v1.0.0*

Compares the abilities obtained under each reference machine with Epoch AI's
Capabilities Index (ECI), with bootstrap intervals, and writes the corresponding figures and
tables.

| reads | writes |
|---|---|
| `ctest_battery.json`, `ctest_metrics.json`, `ctest_results.json`, `eci-leaderboard-*.csv` | figures and LaTeX tables in `figs/` |

**Needs** nothing but the data. CPU only. The ECI files are from Epoch AI, under CC BY 4.0.

## Running it

**Locally.** From the repository root, install the requirements and open the notebook in Jupyter:

```
pip install -r requirements.txt
jupyter lab notebooks/
```

API keys are read from environment variables (see `.env.example`). On the first local run the
notebook copies `data/` into a `work/` folder and runs there, so the released data are never
modified; set `CTEST_WORK_DIR` to use another folder.

**Colab.** Upload the data files to `MyDrive/ctest` and mount Drive; keys go in the Secrets panel.

**Kaggle.** Attach a dataset holding the data files and set `KAGGLE_DATASET` to its name; keys go in
Add-ons, Secrets. With Kaggle credentials the notebook can also push its results back as a new
version of the dataset.


In [ ]:
# Local runs work in a copy of the data, so that data/ stays exactly as released.
# On Colab and Kaggle this cell does nothing.
import os, glob, shutil
if not (os.path.exists("/kaggle") or os.path.exists("/content")):
    _root = os.getcwd()
    if os.path.basename(_root) == "notebooks":
        _root = os.path.dirname(_root)
    try:
        from dotenv import load_dotenv          # optional: keys from a .env file
        load_dotenv(os.path.join(_root, ".env"))
    except ImportError:
        pass
    WORK_DIR = os.path.abspath(os.environ.get("CTEST_WORK_DIR", os.path.join(_root, "work")))
    os.makedirs(WORK_DIR, exist_ok=True)
    for _f in glob.glob(os.path.join(_root, "data", "*")):
        _dst = os.path.join(WORK_DIR, os.path.basename(_f))
        if not os.path.exists(_dst):
            shutil.copy(_f, _dst)
    os.chdir(WORK_DIR)
    print(f"local run: working in {WORK_DIR}")


In [ ]:
# ===========================================================================
# Ctest-Results-ECI
# ===========================================================================
# Ability on this battery, beside a capability measure built somewhere else.
#
# Everything here rests on the certified items and nothing else: those are the
# ones where Kc is the cost of a description the reference machine could run,
# rather than the cost of a copy. A model's ABILITY is where its response
# curve crosses one half, which puts it in bits on the same scale as the items.
#
# The external measure is the Epoch Capability Index. Two versions of it:
# the general one, over every benchmark Epoch tracks, and the same index
# restricted to ARC-AGI-1 and ARC-AGI-2, which are the two closest in kind to
# these items since both ask for a rule inferred from a few examples. The
# restricted index lists fewer models, so the two are also compared on the
# models both list, or the comparison is between different samples rather
# than between two indices.
#
# Reads the battery, the metrics, the results and the two Epoch files. Writes
# one table and five figures per reference machine. Changes nothing.

BUILD = "Ctest-Results-ECI-v1.0.0"
SHOW_TITLES   = False   # titles on the figures; off for the paper, whose captions carry them

# ---- where things are -----------------------------------------------------
IN            = "."          # the ctest_*.json files
ECI_GENERAL   = "eci-leaderboard-general.csv"
ECI_CUSTOM    = "eci-leaderboard-custom-ARG-AGI-1-and-2.csv"
FIGDIR        = "figs"

MACHINES      = [("qwen3-14b-4bit-think",  "qwen3-14b",       "Qwen3-14B"),
                 ("qwen3-8b-4bit-think",   "qwen3-8b",        "Qwen3-8B"),
                 ("phi-4-reasoning",       "phi-4-reasoning", "Phi-4-reasoning")]

N_BOOT        = 600          # resamples for the interval on ability and accuracy
BOOT_SEED     = 7
SCORE         = "full"

# WHICH ROW STANDS FOR A MODEL. A model that declines a share of the items has
# a coverage figure and not a score, so where a fallback exists it is the one
# used, and the table marks it. The Epoch index measures a model that answers,
# and comparing it against a refusal rate would be comparing two things.
PREFER_FALLBACK = True

# The evaluator's name for a model against Epoch's. Only models in both appear.
TO_EPOCH = {
    "gpt-5.6-sol":               "GPT-5.6 Sol",
    "gpt-5-nano":                "GPT-5 nano",
    "claude-opus-5":             "Claude Opus 5",
    "claude-sonnet-5":           "Claude Sonnet 5",
    "claude-haiku-4-5-20251001": "Claude Haiku 4.5",
    "claude-fable-5-1":          "Claude Fable 5.1",
    "gemini-3.8-flash":          "Gemini 3.8 Flash",
    "gemini-3.6-flash":          "Gemini 3.6 Flash",
    "gemini-3.5-flash-lite":     "Gemini 3.5 Flash-Lite",
    "gemini-3.1-flash-lite":     "Gemini 3.1 Flash-Lite",
}

LAB_COLOUR = {"Anthropic": "#d97757", "OpenAI": "#10a37f",
              "Google DeepMind": "#4285f4", "Google": "#4285f4",
              "Alibaba": "#a05195", "Meta AI": "#3b5998"}


In [ ]:
import json, csv, math, os, random, collections, statistics as st
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit

os.makedirs(FIGDIR, exist_ok=True)

def load(name):
    with open(os.path.join(IN, name)) as f:
        return json.load(f)

BAT = {b["id"]: b for b in load("ctest_battery.json")["items"]}
MET = load("ctest_metrics.json")["items"]
ROWS = load("ctest_results.json")["rows"]
print(f"{len(BAT)} items, {len(ROWS)} answer rows, "
      f"{len({r['examinee'] for r in ROWS})} examinees")

def is_valid(b):
    """Recorded two ways, and both mean the same: the seed, run cold a second
    time, reproduced what the first run made. Generation writes `valid`; the
    standalone extension pass writes `repro.seed_cold.verdict`. Reading one of
    them drops half the battery."""
    return (b.get("valid") is True
            or (b.get("repro", {}).get("seed_cold") or {}).get("verdict")
            == "reproduced")

def winner_cert(S):
    """The certification of the program that WON, by p_star, falling back to
    cert_prev. Scanning `programs` for the first passing entry returns
    whichever the dict yields first, which is not the cheapest, and finds
    nothing at all once a re-annotation has moved `cert` aside."""
    P = S.get("programs") or {}
    r = P.get(S.get("p_star")) or {}
    return r.get("cert") or r.get("cert_prev") or {}

def read_eci():
    """Both Epoch files. The general index comes from the general file, which
    lists far more models; the restricted one from the custom file. Taking
    both from the custom file would silently drop every model it does not
    list, which is three of ours."""
    def f(x):
        try:
            return float(x)
        except (TypeError, ValueError):
            return None
    gen, arc = {}, {}
    with open(ECI_GENERAL) as fh:
        for r in csv.DictReader(fh):
            gen[r["Model"]] = (f(r["ECI"]), f(r["ECI CI low"]),
                               f(r["ECI CI high"]), r["Lab"])
    with open(ECI_CUSTOM) as fh:
        for r in csv.DictReader(fh):
            arc[r["Model"]] = (f(r["Custom ECI"]), f(r["Custom ECI CI low"]),
                               f(r["Custom ECI CI high"]), r["Lab"])
            gen.setdefault(r["Model"], (f(r["General ECI"]),
                                        f(r["General ECI CI low"]),
                                        f(r["General ECI CI high"]), r["Lab"]))
    return gen, arc

ECI_GEN, ECI_ARC = read_eci()
print(f"Epoch: {len(ECI_GEN)} models with a general index, "
      f"{len(ECI_ARC)} with the ARC-AGI restriction")


In [ ]:
def score(r):
    """What one attempt scored. `prefix` is the fraction of the k_f terms right
    before the first mistake, which gives partial credit; `full` is all of them
    or nothing."""
    frac = r["n_correct"] / r["k"]
    return frac if SCORE == "prefix" else (1.0 if frac >= 0.999 else 0.0)

def certified_kt(mid, tag):
    """Kt for every item CERTIFIED under this machine, and nothing else.

    An item is here only if it is valid, unquestionable under this machine, and
    some description reproduced the continuation when the machine ran it. On
    the rest Kc is the cost of a copy, and a copy is cheap to execute on the
    items no rule could carry, so pooling the two makes the measure look
    unrelated to difficulty when within the certified set it is not.
    """
    out = {}
    for i, b in BAT.items():
        if not is_valid(b):
            continue
        S = (MET.get(i) or {}).get(mid) or {}
        if S.get("band") in (None, "p_lam"):
            continue
        tau = winner_cert(S).get("tau")
        if S.get("K_cond") is None or not tau:
            continue
        out[i] = S["K_cond"] + math.log2(tau)
    return out

def answers_for(kt, tag):
    """Per examinee and item, the mean over attempts.

    THE PREFIX MUST BE THE ONE THIS MACHINE NEEDS. L_u belongs to a machine, an
    item is administered at the longest any machine needs, and a row answered at
    a shorter prefix answers an easier question. So: per examinee and item, the
    shortest prefix at or above this machine's L_u, and rows below it dropped.
    """
    need = {i: (BAT[i].get(f"n_M_{tag}") or 0) for i in kt}
    best = {}
    for r in ROWS:
        if r["item"] not in kt:
            continue
        n = r.get("n_M")
        if n is None or n < need[r["item"]]:
            continue
        k = (r["examinee"], r["item"])
        if best.get(k) is None or n < best[k]:
            best[k] = n
    acc = collections.defaultdict(lambda: collections.defaultdict(list))
    for r in ROWS:
        if r["item"] in kt and r.get("n_M") == best.get((r["examinee"], r["item"])):
            acc[r["examinee"]][r["item"]].append(score(r))
    return {e: {i: st.mean(v) for i, v in d.items()} for e, d in acc.items()}

def curve(x, th, s):
    """p(x) = 1 / (1 + exp((x - th)/s)), a logistic in BITS.

    th is the difficulty at which the examinee is on the edge and s is the
    width of the transition, both in the units of the measure and free of
    each other. The same form the results notebook uses, so an ability here
    and an ability there are on one scale.

    It replaces 2/(1 + exp((x/b)^k)), a logistic in LOG bits, which was one
    at x = 0 by construction but tied the width to the location: a curve
    could only flatten as it moved right.
    """
    return 1.0 / (1.0 + np.exp(np.clip((np.asarray(x) - th) / s, -50, 50)))

def steepest(xs, ys):
    """Where the fitted curve crosses one half: the ability, in bits.

    The prior at the origin does the work that holding k at 2 or above used to
    do: it keeps the curve near one where an item costs almost nothing,
    without which an examinee is drawn as already unreliable there.
    """
    try:
        # TWICE AS MANY PSEUDO-OBSERVATIONS AT (0, 1) AS THERE ARE ITEMS.
        # An item costing nothing is answered by anyone, and stating that as
        # a prior keeps the curve near 1 at the origin without tying its
        # width to its location. Same prior as the results notebook.
        _n = 2 * len(xs)
        _X = np.concatenate([np.asarray(xs, float), np.zeros(_n)])
        _Y = np.concatenate([np.asarray(ys, float), np.ones(_n)])
        p, _ = curve_fit(curve, _X, _Y,
                         p0=[max(np.median(xs), 1.0), 30.0],
                         bounds=([0.1, 1.0], [1e4, 1e4]), maxfev=40000)
    except Exception:
        return None
    g = np.linspace(0.5, max(xs) * 1.6, 6000)
    return float(g[int(np.argmax(np.abs(np.gradient(curve(g, *p), g))))])

def ability_table(mid, tag):
    """Ability and accuracy per examinee, each with a bootstrap interval.

    THE RESAMPLE IS OVER ITEMS. What is uncertain is which items the battery
    happens to hold, not which attempt an examinee happened to make: another
    battery built the same way would hold different items, and that is the
    variation a reader should see.
    """
    kt = certified_kt(mid, tag)
    ans = answers_for(kt, tag)
    items = sorted(kt)
    rng = random.Random(BOOT_SEED)
    out = {}
    for e, d in ans.items():
        if len(d) < len(items):
            continue                       # not asked every certified item
        xs = np.array([kt[i] for i in items])
        ys = np.array([d[i] for i in items])
        a0 = steepest(xs, ys)
        if a0 is None:
            print(f"    {e}: the curve would not fit, left out")
            continue
        accs, abis = [], []
        for _ in range(N_BOOT):
            idx = [rng.randrange(len(items)) for _ in items]
            accs.append(float(ys[idx].mean()))
            p = steepest(xs[idx], ys[idx])
            if p is not None:
                abis.append(p)
        accs.sort(); abis.sort()
        q = lambda v, p: v[max(0, min(len(v) - 1, int(p * len(v))))]
        out[e] = dict(acc=float(ys.mean()), acc_lo=q(accs, .025), acc_hi=q(accs, .975),
                      ability=a0, ab_lo=q(abis, .025), ab_hi=q(abis, .975))
    return out, len(items)


In [ ]:
def pearson(x, y):
    mx, my = st.mean(x), st.mean(y)
    d = math.sqrt(sum((a - mx) ** 2 for a in x) * sum((b - my) ** 2 for b in y))
    return sum((a - mx) * (b - my) for a, b in zip(x, y)) / d if d else 0.0

def spearman(x, y):
    def rank(v):
        s = sorted(range(len(v)), key=lambda i: v[i])
        o = [0] * len(v)
        for p, i in enumerate(s):
            o[i] = p
        return o
    return pearson(rank(x), rank(y))

def paired(ability, index):
    """Our numbers beside one Epoch index, for the models in both.

    Where a fallback row exists it stands for the model: Epoch measures a model
    that answers, and the plain row of a model that declines a share of the
    items is a coverage figure. The substitution is marked so a reader can see
    which rows it applied to.
    """
    out = []
    for key, epoch_name in TO_EPOCH.items():
        if epoch_name not in index or index[epoch_name][0] is None:
            continue
        fb = f"{key}-fallback"
        use = fb if (PREFER_FALLBACK and fb in ability) else key
        if use not in ability:
            continue
        v, lo, hi, lab = index[epoch_name]
        out.append(dict(key=use, name=epoch_name, lab=lab, fallback=use.endswith("-fallback"),
                        eci=v, eci_lo=lo, eci_hi=hi, **ability[use]))
    return out

def _short(name):
    """The short labels the tables use, from Epoch's display names. Written
    out rather than derived: Epoch writes `GPT-5.6 Sol' where the results
    file writes `gpt-5.6-sol', and one map is clearer than two rules."""
    M = {"GPT-5.6 Sol": "gpt5sol", "GPT-5 nano": "gpt5na",
         "Claude Opus 5": "opus", "Claude Sonnet 5": "sonnet",
         "Claude Haiku 4.5": "haiku", "Claude Fable 5.1": "fable",
         "Gemini 3.8 Flash": "g3.8F", "Gemini 3.6 Flash": "g3.6F",
         "Gemini 3.5 Flash-Lite": "g3.5FL", "Gemini 3.1 Flash-Lite": "g3.1FL",
         "Gemma 4 26B": "gemma4", "OLMo 2 7B": "olmo2",
         "Qwen2.5 1.5B": "q1.5B"}
    return M.get(name, name)

def scatter(T, xk, xlo, xhi, yk, ylo, yhi, xlabel, ylabel, title, path):
    fig, ax = plt.subplots(figsize=(8.0, 5.2))
    for r in T:
        # GOOGLE AND GOOGLE DEEPMIND ARE ONE VENDOR here. Epoch lists both
        # and the split says nothing about the models.
        _lab = "Google" if r["lab"].startswith("Google") else r["lab"]
        c = LAB_COLOUR.get(_lab, "0.4")
        ax.plot([r[xlo], r[xhi]], [r[yk], r[yk]], color=c, lw=1.1, alpha=.35, zorder=2)
        ax.plot([r[xk], r[xk]], [r[ylo], r[yhi]], color=c, lw=1.1, alpha=.35, zorder=2)
        ax.scatter([r[xk]], [r[yk]], s=74, color=c, zorder=3,
                   edgecolor="white", lw=1.0)
        ax.annotate(_short(r["name"]), (r[xk], r[yk]),
                    textcoords="offset points", xytext=(0, 11), ha="center",
                    fontsize=8.2, color="0.25")
    xs = [r[xk] for r in T]; ys = [r[yk] for r in T]
    mx, my = st.mean(xs), st.mean(ys)
    b = sum((a - mx) * (c - my) for a, c in zip(xs, ys)) / sum((a - mx) ** 2 for a in xs)
    a0 = my - b * mx
    lo, hi = min(xs), max(xs); pad = (hi - lo) * 0.12
    ax.plot([lo - pad, hi + pad], [a0 + b * (lo - pad), a0 + b * (hi + pad)],
            color="0.55", ls="--", lw=1.1, zorder=1)
    ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    # THE CORRELATIONS GO IN THE PLOT, not the title, so the figure carries
    # them when the title is stripped for the paper.
    ax.text(0.02, 0.03,
            f"Pearson {pearson(xs, ys):+.2f}\nSpearman {spearman(xs, ys):+.2f}",
            transform=ax.transAxes, fontsize=9, color="0.25",
            va="bottom", ha="left",
            bbox=dict(boxstyle="round,pad=0.35", fc="white", ec="0.8", lw=.8))
    SHOW_TITLES and ax.set_title(f"{title}\nbars: 95 per cent, ours from {N_BOOT} "
                        f"bootstrap resamples over items", fontsize=9.5)
    ax.grid(alpha=.25)
    labs = sorted({("Google" if r["lab"].startswith("Google") else r["lab"])
                   for r in T})
    H = [Line2D([], [], marker="o", ls="", color=LAB_COLOUR.get(l, "0.4"), label=l)
         for l in labs]
    # INSIDE, upper left: three or four vendor names fit the empty corner,
    # where an outside legend cost a quarter of the width.
    ax.legend(handles=H, loc="upper left", frameon=True, framealpha=.9,
              edgecolor="0.8", fontsize=8)
    fig.tight_layout()
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    print(f"  wrote {path}")


In [ ]:
CAPTION = (
 r"Ability is read off the response curve. For each examinee the score on each "
 r"item is fitted against $\widehat{Kt}_\M(x_{n+1:n+k} \mid x_{1:n})$ with "
 r"$p(x) = 1/(1 + e^{(x-\theta)/s})$, a logistic in bits fitted with twice as "
 r"many pseudo-observations at $(0,1)$ as there are items, which keeps it near "
 r"one where an item costs nothing and falls as the items "
 r"cost more to describe; the ability reported is the $x$ at which that curve "
 r"crosses one half, which is $\theta$. It is a difficulty in bits, on the same "
 r"scale as the items, and "
 r"reads as the point at which an examinee stops being reliable. Accuracy is the "
 r"mean of the same scores. The interval on both is the 2.5th and 97.5th "
 r"percentile of BOOT bootstrap resamples OVER ITEMS: what is uncertain is which "
 r"items a battery built this way happens to hold, not which attempt an examinee "
 r"happened to make. ECI is the Epoch Capability Index, built from public "
 r"benchmarks, with its own interval from that source and independent of this "
 r"battery. ECI-ARC is the same index restricted to ARC-AGI-1 and ARC-AGI-2, the "
 r"two benchmarks closest in kind to these items, both asking for a rule inferred "
 r"from a few examples; it lists fewer models and a dash marks those. Where a "
 r"model declines a share of the items the version that answers stands for it, "
 r"marked with an asterisk, since a refusal is a coverage figure and not a score.")

def write_table(G, A, n_items, shown, path):
    """One table, both indices, and the correlations twice over.

    The second set is on the models ECI-ARC also lists. Comparing an index over
    ten models with an index over seven says as much about which three were
    dropped as about the indices: here the general one rises from +0.75 to
    +0.82 on the smaller sample, so most of what looked like the restricted
    index agreeing better was the sample changing.
    """
    by_key = {r["key"]: r for r in G}
    arc = {r["key"]: r for r in A}
    order = sorted(by_key, key=lambda k: -by_key[k]["ability"])
    L = [r"\begin{table}[t]\centering\small",
         r"\caption{Ability and accuracy on the " + str(n_items) +
         r" items certified under " + shown + r", beside two external capability "
         r"measures. " + CAPTION.replace("BOOT", str(N_BOOT)) + r"}",
         r"\label{tab:ability-" + shown.lower() + r"}",
         r"\begin{tabular}{@{}llrrrr@{}}\toprule",
         r"model & lab & ability [95\%] & accuracy [95\%] & ECI [95\%] & "
         r"ECI-ARC [95\%] \\", r"\midrule"]
    for k in order:
        r = by_key[k]; a = arc.get(k)
        L.append(f"{r['name']}{'*' if r['fallback'] else ''} & "
                 f"{r['lab'].replace('Google DeepMind', 'GDM')} & "
                 f"{r['ability']:.0f} [{r['ab_lo']:.0f}, {r['ab_hi']:.0f}] & "
                 f"{r['acc'] * 100:.1f} [{r['acc_lo'] * 100:.0f}, {r['acc_hi'] * 100:.0f}] & "
                 f"{r['eci']:.1f} [{r['eci_lo']:.1f}, {r['eci_hi']:.1f}] & "
                 + (f"{a['eci']:.1f} [{a['eci_lo']:.1f}, {a['eci_hi']:.1f}]"
                    if a else r"---") + r" \\")
    both = [k for k in order if k in arc]
    ab_a = [by_key[k]["ability"] for k in order]
    ac_a = [by_key[k]["acc"] for k in order]
    g_a = [by_key[k]["eci"] for k in order]
    ab_b = [by_key[k]["ability"] for k in both]
    ac_b = [by_key[k]["acc"] for k in both]
    g_b = [by_key[k]["eci"] for k in both]
    a_b = [arc[k]["eci"] for k in both]
    foot = (r"\multicolumn{6}{@{}p{0.95\textwidth}@{}}{\footnotesize "
            r"Pearson, Spearman in brackets. Over all " + str(len(order)) + r" models: "
            f"ability against accuracy {pearson(ab_a, ac_a):+.2f} ({spearman(ab_a, ac_a):+.2f}), "
            f"ability against ECI {pearson(ab_a, g_a):+.2f} ({spearman(ab_a, g_a):+.2f}), "
            f"accuracy against ECI {pearson(ac_a, g_a):+.2f} ({spearman(ac_a, g_a):+.2f}). "
            r"Over the " + str(len(both)) + r" models ECI-ARC also lists, so that the two "
            r"indices are compared on one sample: "
            f"ability against ECI {pearson(ab_b, g_b):+.2f} ({spearman(ab_b, g_b):+.2f}), "
            f"ability against ECI-ARC {pearson(ab_b, a_b):+.2f} ({spearman(ab_b, a_b):+.2f}), "
            f"accuracy against ECI {pearson(ac_b, g_b):+.2f} ({spearman(ac_b, g_b):+.2f}), "
            f"accuracy against ECI-ARC {pearson(ac_b, a_b):+.2f} ({spearman(ac_b, a_b):+.2f})."
            + r"} \\")
    L += [r"\midrule", foot, r"\bottomrule", r"\end{tabular}", r"\end{table}"]
    open(path, "w").write("\n".join(L) + "\n")
    print(f"  wrote {path}")
    return dict(all_n=len(order), both_n=len(both),
                ab_eci_all=pearson(ab_a, g_a), ab_eci_both=pearson(ab_b, g_b),
                ab_arc_both=pearson(ab_b, a_b))

for mid, tag, shown in MACHINES:
    print(f"\n=== {shown}")
    ability, n_items = ability_table(mid, tag)
    G, A = paired(ability, ECI_GEN), paired(ability, ECI_ARC)
    print(f"  {n_items} certified items, {len(ability)} examinees fitted, "
          f"{len(G)} matched to ECI, {len(A)} to ECI-ARC")
    s = write_table(G, A, n_items, shown, f"{FIGDIR}/tab_ability_{tag}.tex")
    KT = r"ability: $\theta$, the $Kt$ at which accuracy crosses one half, bits"
    AC = "accuracy on the certified items"
    ttl = f"({{}} models, {n_items} certified items under {shown})"
    for T, lbl, sfx in ((G, "ECI", "eci"), (A, "ECI-ARC", "eciarc")):
        scatter(T, "ability", "ab_lo", "ab_hi", "eci", "eci_lo", "eci_hi",
                KT, lbl, ttl.format(len(T)), f"{FIGDIR}/fig_tri_ability_{sfx}_{tag}.png")
        scatter(T, "acc", "acc_lo", "acc_hi", "eci", "eci_lo", "eci_hi",
                AC, lbl, ttl.format(len(T)), f"{FIGDIR}/fig_tri_accuracy_{sfx}_{tag}.png")
    scatter(G, "ability", "ab_lo", "ab_hi", "acc", "acc_lo", "acc_hi",
            KT, AC, ttl.format(len(G)), f"{FIGDIR}/fig_tri_ability_accuracy_{tag}.png")
    print(f"  ability against ECI {s['ab_eci_all']:+.2f} over {s['all_n']}, "
          f"{s['ab_eci_both']:+.2f} over the {s['both_n']} ECI-ARC lists; "
          f"against ECI-ARC {s['ab_arc_both']:+.2f}")

print("\nNothing was changed. The tables and figures are in " + FIGDIR + ".")
